# Prediction tutorial: energy demand or generation

The root `prediction` package is a small forecasting toolkit for an energy value observed over time. The **target** can be consumption (also called demand) or production (also called generation). This tutorial builds tiny synthetic examples so it needs no downloaded data or extra dependencies.

Some public names remain consumption-specific for compatibility with existing code. In particular, `predict_consumption`, the default target `Consumption`, and output columns such as `Predicted_Consumption` may still appear even when the configured target represents generation.

## Terms and workflow

- **Timestamp:** when an observation was measured.
- **Target:** the numeric value to forecast, such as `Consumption` or `Production`.
- **Feature:** a numeric input that helps the model learn a pattern. Here, calendar features represent hour, weekday, and season as repeating cycles.
- **Train/test split:** older rows train the model; newer rows are held back to estimate performance on unseen times.
- **Model:** the fitted relationship between features and the target.
- **Evaluation:** a score on the held-out test rows. We use mean absolute error (MAE), where lower is better.
- **Prediction:** a target estimate for a requested timestamp.

| Stage | Input | Output |
|---|---|---|
| Prepare | timestamp + target rows | cleaned chronological data |
| Describe time | timestamps | numeric calendar features |
| Split and train | older feature/target rows | fitted model |
| Evaluate | newer feature/target rows | held-out MAE |
| Predict | a future timestamp | forecast target value |

## 1. Create a small dataset

The package expects CSV data with a `Datetime` column and one numeric target column. The two frames below show the same timestamps with different targets. Files are written only inside a temporary directory because the package APIs accept CSV paths.

In [ ]:
from pathlib import Path
import tempfile

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

from prediction.predicting_consumption_model import (
    create_predictor,
    default_time_features,
    detect_features,
    generate_day_predictions,
    load_data,
    predict_consumption,
    train_gradient_boosting,
)

temporary_workspace = tempfile.TemporaryDirectory()
temporary_path = Path(temporary_workspace.name)

In [ ]:
timestamps = pd.date_range("2025-01-01", periods=14 * 24, freq="h")
hour = timestamps.hour.to_numpy()
weekday = timestamps.weekday.to_numpy()

consumption_data = pd.DataFrame({
    "Datetime": timestamps,
    "Consumption": 24 + 5 * np.sin(2 * np.pi * (hour - 7) / 24) + 2 * (weekday < 5),
})
production_data = pd.DataFrame({
    "Datetime": timestamps,
    "Production": 18 * np.maximum(0, np.sin(np.pi * (hour - 6) / 12)),
})

consumption_csv = temporary_path / "consumption.csv"
production_csv = temporary_path / "production.csv"
consumption_data.to_csv(consumption_csv, index=False)
production_data.to_csv(production_csv, index=False)

consumption_data.head(3)

A real file follows the same shape: one row per timestamp and a target name that matches the `target_col` argument. Extra numeric columns can also become model features, so include only values that will be known when predictions are made.

## 2. Compatibility path: default `Consumption`

`create_predictor` performs loading, feature creation, feature detection, and training. With no `target_col`, it keeps the historical `Consumption` default. The trained object can predict one timestamp or a sequence of full days.

In [ ]:
consumption_predictor = create_predictor(consumption_csv)
one_consumption_forecast = consumption_predictor.predict("2025-01-15", "08:00")
consumption_day = consumption_predictor.predict_days("2025-01-15", num_days=1)

print(f"08:00 consumption forecast: {one_consumption_forecast:.2f}")
consumption_day.head(3)

Notice that the daily result uses `Predicted_Consumption`. That name is intentionally stable for older callers.

## 3. General path: a non-default `Production` target

The lower-level functions make each forecasting step visible. We first load and clean `Production`, derive numeric time features, and identify the feature columns. Then we split chronologically: the first 80% trains the model and the final 20% remains unseen until evaluation.

In [ ]:
production = load_data(production_csv, target_col="Production")
production = default_time_features(production)
production_features = detect_features(production, target_col="Production")

split_index = int(len(production) * 0.8)
production_train = production.iloc[:split_index]
production_test = production.iloc[split_index:]

print(f"Features: {production_features}")
print(f"Train rows: {len(production_train)}; test rows: {len(production_test)}")

Training fits a gradient-boosting regression model. Evaluation compares its forecasts with the held-out production values; MAE is reported in the same units as the target.

In [ ]:
production_model = train_gradient_boosting(
    production_train,
    production_features,
    target_col="Production",
)
test_forecasts = production_model.predict(production_test[production_features])
production_mae = mean_absolute_error(production_test["Production"], test_forecasts)

print(f"Held-out production MAE: {production_mae:.3f}")

Finally, use the same feature function for new timestamps. The function is still named `predict_consumption`, but here it returns a **production** forecast because the model was trained on `Production`. Likewise, `generate_day_predictions` keeps the compatibility column `Predicted_Consumption`; interpret it as the configured target.

In [ ]:
noon_production = predict_consumption(
    production_model,
    production_features,
    default_time_features,
    "2025-01-15",
    "12:00",
)
production_day = generate_day_predictions(
    production_model,
    production_features,
    default_time_features,
    start_date="2025-01-15",
    num_days=1,
)

print(f"12:00 production forecast: {noon_production:.2f}")
production_day[["Date", "Time", "Predicted_Consumption"]].iloc[10:15]

## 4. What to change for real data

1. Replace the temporary CSV with your timestamped measurements.
2. Set `target_col` to the exact demand or generation column name.
3. Add only features available at forecast time, such as calendar or weather forecasts.
4. Keep the split chronological and judge candidate models on later, unseen timestamps.
5. Retrain on appropriate recent history before producing operational forecasts.

In [ ]:
temporary_workspace.cleanup()